# Notebook 1 — Benchmark

Four models of the same family, twelve prompts, three repetitions each: 144 measured calls.
For every call this notebook records the energy drawn by the machine, the latency, the number
of tokens produced, and whether the answer is correct. It then grades quality, fits the
scaling relation and runs the statistical tests.

**Why one family and one quantization.** All four models are Qwen2.5 in Q4_K_M. Changing only
the parameter count means any difference measured here is attributable to model size and not
to a different architecture, tokenizer or precision.

**Running it.** Execute the cells in order. The measurement cell takes two to four hours; if
it is interrupted, run that same cell again — it reads what is already in
`outputs/benchmark_runs.csv` and carries on from there. Close other applications first: the
measurement is of the whole machine, so a video call lands in the figures.

**Prerequisites.** Ollama running locally, and five models pulled:

```
ollama pull qwen2.5:0.5b
ollama pull qwen2.5:1.5b
ollama pull qwen2.5:3b
ollama pull qwen2.5:7b
ollama pull llama3.1:8b
```

In [ ]:
import random

import numpy as np
import pandas as pd

from core import ask, idle_watts, describe_machine, save_table, OUTPUTS, SEED
from prompt_bank import PROMPT_BANK

MODELS = ["qwen2.5:0.5b", "qwen2.5:1.5b", "qwen2.5:3b", "qwen2.5:7b"]
PARAMETERS_B = {"qwen2.5:0.5b": 0.5, "qwen2.5:1.5b": 1.5, "qwen2.5:3b": 3.0, "qwen2.5:7b": 7.0}
JUDGES = ["llama3.1:8b", "qwen2.5:7b"]
REPETITIONS = 3

print("Machine")
for key, value in describe_machine().items():
    print(f"  {key:15} {value}")

checkable = sum(1 for prompt in PROMPT_BANK if prompt["check"] is not None)
print(f"\nPrompts: {len(PROMPT_BANK)}, of which {checkable} carry a deterministic check")
print(f"Runs: {len(MODELS)} models x {len(PROMPT_BANK)} prompts x {REPETITIONS} repetitions "
      f"= {len(MODELS) * len(PROMPT_BANK) * REPETITIONS}")

## 1. Idle power, before the campaign

The machine draws power even when it is doing nothing, and that draw is inside every
measurement below — the figures are the cost of a request on this machine, not the marginal
cost of the computation. The idle draw is measured here and again at the end, so that any
change in the machine's state during the session becomes visible instead of silently
shifting the numbers.

In [ ]:
idle_before = idle_watts()
print(f"idle power before: {idle_before:.2f} W")

## 2. Measure

Two precautions, both of which change the result if they are omitted.

**The model is unloaded before every call**, so each run pays its own loading cost. Without
this, whichever model happens to run second finds itself already in memory and looks cheaper
than it is.

**The order of the 144 runs is shuffled** with a fixed seed, so that no model systematically
benefits from a machine that has warmed up or cooled down.

The CSV is rewritten after every single run. That is what makes the cell resumable.

In [ ]:
RUNS_FILE = OUTPUTS / "benchmark_runs.csv"

schedule = [(model, prompt, repeat)
            for model in MODELS
            for prompt in PROMPT_BANK
            for repeat in range(REPETITIONS)]
random.Random(SEED).shuffle(schedule)

if RUNS_FILE.exists():
    measured = pd.read_csv(RUNS_FILE)
    already_done = set(zip(measured.model, measured.prompt_id, measured.repeat))
    print(f"{len(measured)} runs already in the file, continuing from there\n")
else:
    measured = pd.DataFrame()
    already_done = set()

new_rows = []
for position, (model, prompt, repeat) in enumerate(schedule, start=1):
    if (model, prompt["id"], repeat) in already_done:
        continue

    result = ask(model, prompt["prompt"])
    check = prompt["check"]
    new_rows.append({
        "model": model,
        "prompt_id": prompt["id"],
        "tier": prompt["tier"],
        "repeat": repeat,
        "parameters_b": PARAMETERS_B[model],
        "energy_wh": result["energy_wh"],
        "duration_s": result["duration_s"],
        "load_s": result["load_s"],
        "tokens_in": result["tokens_in"],
        "tokens_out": result["tokens_out"],
        "answer": result["answer"],
        "objective_check": check(result["answer"]) if check else np.nan,
    })

    pd.concat([measured, pd.DataFrame(new_rows)], ignore_index=True).to_csv(
        RUNS_FILE, index=False, encoding="utf-8-sig")

    print(f"[{position:3}/{len(schedule)}] {model:14} {prompt['id']:22} "
          f"{result['energy_wh']:7.4f} Wh  {result['duration_s']:6.1f} s  "
          f"{result['tokens_out']:4} tokens")

runs = pd.read_csv(RUNS_FILE)
print(f"\n{len(runs)} runs in {RUNS_FILE}")

## 3. Idle power, after the campaign

In [ ]:
idle_after = idle_watts()
drift = 100 * (idle_after - idle_before) / idle_before
print(f"idle power after: {idle_after:.2f} W   drift {drift:+.1f} %")

if abs(drift) > 15:
    print("\nThe machine was not in the same state at the start and at the end of the session.")
    print("Runs measured early and late are not strictly comparable. Report this drift,")
    print("or delete outputs/benchmark_runs.csv and measure again on a quiet machine.")
else:
    print("\nThe drift is small: early and late runs are comparable.")

## 4. Answers that can be checked without a judge

Seven of the twelve prompts have a single verifiable answer — a capital city, three fields
extracted from an invoice, a category, a translation, two calculations, one function. For
these, correctness is decided by a deterministic check written in advance, with no model
involved. The other five ask for a summary, an analysis or a recommendation: there is no
single correct answer, so only the judges can speak about them.

One of those checks, on the code-writing prompt, runs the generated function in a separate
Python process with a twenty-second timeout and asserts its output. That is the only place in
these notebooks where model-generated code is executed.

In [ ]:
for prompt in PROMPT_BANK:
    verifiable = "deterministic check" if prompt["check"] else "judges only"
    print(f"{prompt['id']:22} {prompt['tier']}   {verifiable}")

## 5. Quality, scored by two judges

Each answer is scored from 1 to 5 on five criteria by two models from different families —
llama3.1:8b and qwen2.5:7b — so that their agreement can be measured rather than assumed.
Section 7 reports that agreement.

The judges see the question, the reference answer and the answer to score, but never the name
of the model that produced it. Their own energy consumption is recorded for information and is
excluded from every energy figure in this work: they are measuring instruments, not part of
any pipeline an organization would run.

The prompts, the reference answers and the scoring rubric are in French because the use case
studied is a French-speaking organization. The analysis and the figures are in English.

In [ ]:
import hashlib
import json as jsonlib

RUBRIC = ("Note la réponse de 1 à 5 sur cinq critères : exactitude, clarté, structure, "
          "complétude, utilité métier. Réponds UNIQUEMENT en JSON avec les clés "
          "exactitude, clarte, structure, completude, utilite, commentaire.")

CRITERIA = ["exactitude", "clarte", "structure", "completude", "utilite"]

# The judges grade the first repetition of each model-prompt pair: 4 x 12 = 48 answers.
to_grade = runs[runs.repeat == 0].copy()
to_grade["answer_hash"] = [hashlib.sha1(str(answer).encode()).hexdigest()[:12]
                           for answer in to_grade.answer]

references = {prompt["id"]: prompt["reference"] for prompt in PROMPT_BANK}

judge_rows = []
for _, row in to_grade.iterrows():
    for judge in JUDGES:
        question = (f"{RUBRIC}\n\nQUESTION :\n{row.prompt_id}"
                    f"\n\nRÉPONSE DE RÉFÉRENCE :\n{references[row.prompt_id]}"
                    f"\n\nRÉPONSE À NOTER :\n{row.answer}")
        result = ask(judge, question, max_new_tokens=400, unload_first=False)

        # The judge is asked for JSON; take the outermost braces and parse what is inside.
        text = result["answer"]
        score, comment = np.nan, ""
        start, end = text.find("{"), text.rfind("}")
        if start >= 0 and end > start:
            try:
                parsed = jsonlib.loads(text[start:end + 1])
                score = float(np.mean([float(parsed[criterion]) for criterion in CRITERIA]))
                comment = str(parsed.get("commentaire", ""))
            except (ValueError, KeyError, TypeError):
                pass          # unreadable answer: no score, counted below

        judge_rows.append({"judge": judge, "model": row.model, "prompt_id": row.prompt_id,
                           "tier": row.tier, "answer_hash": row.answer_hash,
                           "score": score, "comment": comment,
                           "judge_energy_wh": result["energy_wh"]})

judges = save_table(pd.DataFrame(judge_rows), "judge_scores")
print(f"{judges.score.isna().sum()} of {len(judges)} scores could not be parsed")

## 6. Headline table

One row per model and tier: what a request costs, how long it takes, how good it is.

In [ ]:
summary = (runs.groupby(["model", "tier"], as_index=False)
               .agg(runs=("energy_wh", "size"),
                    energy_wh=("energy_wh", "mean"),
                    latency_s=("duration_s", "mean"),
                    tokens_out=("tokens_out", "mean"),
                    objective_accuracy=("objective_check", "mean")))

quality = judges.groupby(["model", "tier"], as_index=False).score.mean()
summary = summary.merge(quality.rename(columns={"score": "judged_quality"}),
                        on=["model", "tier"], how="left")

save_table(summary.round(4), "summary_by_model_tier")
summary.round(3)

## 7. How energy responds to model size

An ordinary least-squares regression on logarithms, so that each coefficient reads directly
as an elasticity: a 1 % increase in the input gives a *coefficient* percent increase in energy.
Two inputs are separated because they are often conflated — the number of parameters (how
expensive each token is to produce) and the number of tokens produced (how long the answer is).

`cov_type="HC3"` asks for standard errors that do not assume the variance is constant across
model sizes, which it plainly is not here.

In [ ]:
import statsmodels.formula.api as smf

per_cell = (runs.groupby(["prompt_id", "model"], as_index=False)
                .agg(energy_wh=("energy_wh", "mean"),
                     tokens_out=("tokens_out", "mean"),
                     parameters_b=("parameters_b", "first")))
per_cell = per_cell[(per_cell.energy_wh > 0) & (per_cell.tokens_out > 0)].copy()

per_cell["log_energy"] = np.log(per_cell.energy_wh)
per_cell["log_parameters"] = np.log(per_cell.parameters_b)
per_cell["log_tokens"] = np.log(per_cell.tokens_out)

scaling = smf.ols("log_energy ~ log_parameters + log_tokens",
                  data=per_cell).fit(cov_type="HC3")
print(scaling.summary())

## 8. Are the differences between models real?

Twelve prompts is a small sample and the energy distribution is skewed, so the tests used here
work on ranks and assume nothing about the shape of that distribution.

- **Kruskal-Wallis** asks a single question: is at least one of the four models different from
  the others? It ignores the fact that the same twelve prompts were given to all four.
- **Friedman** asks the same question while keeping each prompt as its own block, which is the
  design actually used here — every model answered every prompt.
- Then **each of the six pairs** is compared with a Wilcoxon signed-rank test on the paired
  per-prompt means. Six tests on the same data give six chances of a false positive, so
  **Holm** raises the bar each test has to clear. The `p Holm` column is the one to read.

The column `median B/A` is the plain-language version of each comparison: the median ratio of
the energy of model B to that of model A, across the twelve prompts.

In [ ]:
from scipy import stats

per_prompt = runs.groupby(["prompt_id", "model"]).energy_wh.mean().unstack()
per_prompt = per_prompt[MODELS].dropna()

statistic, p_kruskal = stats.kruskal(*[runs.loc[runs.model == m, "energy_wh"] for m in MODELS])
chi2, p_friedman = stats.friedmanchisquare(*[per_prompt[m] for m in MODELS])
print(f"Kruskal-Wallis   H    = {statistic:8.3f}   p = {p_kruskal:.3e}")
print(f"Friedman         chi2 = {chi2:8.3f}   p = {p_friedman:.3e}   "
      f"({len(per_prompt)} prompts, paired)\n")


def holm(p_values):
    """Holm-Bonferroni correction: raises each p-value for the number of tests made."""
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values))
    running_max = 0.0
    for rank, index in enumerate(order):
        running_max = max(running_max, (len(p_values) - rank) * p_values[index])
        adjusted[index] = min(running_max, 1.0)
    return adjusted


comparisons = []
for i, model_a in enumerate(MODELS):
    for model_b in MODELS[i + 1:]:
        comparisons.append({
            "model A": model_a,
            "model B": model_b,
            "median B/A": np.median(per_prompt[model_b] / per_prompt[model_a]),
            "p Wilcoxon": stats.wilcoxon(per_prompt[model_a], per_prompt[model_b]).pvalue,
        })

pairs = pd.DataFrame(comparisons)
pairs["p Holm"] = holm(pairs["p Wilcoxon"])
pairs["differ at 5 %"] = np.where(pairs["p Holm"] < 0.05, "yes", "no")
save_table(pairs, "pairwise_tests")
pairs.round(5)

## 9. Do the two judges agree?

If they disagree, the quality figures are an opinion rather than a measurement — and that has
to be visible. Spearman's rho is used rather than a correlation on the values themselves
because the scores are ordinal.

In [ ]:
paired = (judges[judges.score.notna()]
          .pivot_table(index="answer_hash", columns="judge", values="score", aggfunc="first")
          .dropna())

if paired.shape[1] < 2:
    print("The judges cannot be compared: fewer than two of them returned readable JSON.")
    print(f"Unreadable scores: {judges.score.isna().sum()} of {len(judges)}.")
    print("Run section 5 again. Nothing else in this notebook depends on this comparison.")
else:
    first_judge, second_judge = paired.columns[:2]
    rho, p_rho = stats.spearmanr(paired[first_judge], paired[second_judge])

    print(f"{first_judge}  vs  {second_judge}")
    print(f"answers scored by both   n = {len(paired)}")
    print(f"Spearman rho             {rho:.4f}   p = {p_rho:.3e}")
    print(f"mean difference          {(paired[first_judge] - paired[second_judge]).mean():+.4f}"
          f" points")
    print("\nWith a single judge this check is impossible: its scores would have to be")
    print("taken on trust. That is the reason for using two.")

## 10. Figures

Four charts, one per question. In all of them the bars run light to dark with model size, so
the darkest bar is always the largest model.

In [ ]:
from core import MODEL_COLOURS, TIER_LABELS, model_label, new_figure, save_figure

tiers = sorted(runs.tier.unique())


def grouped_bars(values, title, subtitle, ylabel, name):
    """values: a dict keyed by (model, tier). One group of four bars per tier."""
    figure, axes = new_figure(title, subtitle, ylabel)
    width = 0.8 / len(MODELS)
    positions = np.arange(len(tiers))

    for index, model in enumerate(MODELS):
        heights = [values.get((model, tier), np.nan) for tier in tiers]
        axes.bar(positions + index * width - 0.4 + width / 2, heights, width * 0.9,
                 label=model_label(model), color=MODEL_COLOURS[model], zorder=3)

    axes.set_xticks(positions)
    axes.set_xticklabels([TIER_LABELS.get(tier, tier) for tier in tiers])
    save_figure(figure, axes, name)


grouped_bars(runs.groupby(["model", "tier"]).energy_wh.mean().to_dict(),
             "Energy per request, by model and task tier",
             f"Qwen2.5 family, Q4_K_M quantization, {len(runs)} measured runs on this machine.",
             "Energy per request (Wh)", "F1_energy")

grouped_bars(runs.groupby(["model", "tier"]).duration_s.mean().to_dict(),
             "End-to-end latency per request, by model and task tier",
             "Measured from sending the prompt to receiving the complete answer.",
             "Latency (s)", "F2_latency")

grouped_bars(judges.groupby(["model", "tier"]).score.mean().to_dict(),
             "Judged quality, by model and task tier",
             "Mean of five criteria, averaged over two independent judges.",
             "Judged quality (1 to 5)", "F3_quality")

grouped_bars(runs.groupby(["model", "tier"]).objective_check.mean().to_dict(),
             "Objective accuracy on the verifiable prompts",
             "Deterministic checks written in advance. No judge involved.",
             "Share of answers that pass the check", "F4_accuracy")

## 11. Output of this notebook

In `outputs/`:

| File | What it holds |
|---|---|
| `benchmark_runs.csv` | one row per measured call: 144 rows |
| `judge_scores.csv` | one row per answer and judge: 96 rows |
| `summary_by_model_tier.csv` | the headline table of section 6 |
| `pairwise_tests.csv` | the six pairwise comparisons of section 8 |
| `figures/F1_energy.png` … `F4_accuracy.png` | the four charts |